In [105]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [106]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [107]:
# Generate a random bit using quantum measurement
def quantum_random_bit():

    qc = QuantumCircuit(1, 1)

    qc.h(0)

    qc.measure(0, 0)

    simulator = BasicSimulator()

    compiled = transpile(qc, simulator)

    result = simulator.run(compiled, shots=1).result()

    counts = result.get_counts()

    return int(list(counts.keys())[0])

In [108]:
# Generate a sequence of random bits
def generate_random_bits(n):

    bits = []

    for i in range(n):
        bits.append(quantum_random_bit())

    return bits

In [109]:
# Encode a bit using either the standard or diagonal basis
def prepare_qubit(bit, basis):

    qc = QuantumCircuit(1, 1)

    # Encode bit value
    if bit == 1:
        qc.x(0)

    # Convert to diagonal basis
    if basis == 1:
        qc.h(0)

    return qc

In [110]:
# Measure a qubit using the chosen basis
def measure_qubit(qc, basis):

    # Convert diagonal basis back to standard basis
    if basis == 1:
        qc.h(0)

    qc.measure(0, 0)

    simulator = BasicSimulator()

    compiled = transpile(qc, simulator)

    result = simulator.run(compiled, shots=1).result()

    counts = result.get_counts()

    return int(list(counts.keys())[0])

In [111]:
n = 20

# =========================
# Alice's part (Sender)
# =========================

## Alice generates random bits and bases
alice_bits = generate_random_bits(n)
alice_bases = generate_random_bits(n)

qubits = []

for i in range(n):

    qubit = prepare_qubit(
        alice_bits[i],
        alice_bases[i]
    )

    qubits.append(qubit)


In [112]:
# =========================
# Eve's part (Attacker)
# =========================

## Eve randomly chooses bases
eve_bases = generate_random_bits(n)

eve_bits = []

resent_qubits = []

for i in range(n):

    # Eve measures Alice's qubit
    eve_bit = measure_qubit(
        qubits[i],
        eve_bases[i]
    )

    eve_bits.append(eve_bit)

    # Eve resends a new qubit to bob
    resent_qubit = prepare_qubit(
        eve_bit,
        eve_bases[i]
    )

    resent_qubits.append(resent_qubit)

In [113]:
# =========================
# Bob's part (Receiver)
# =========================

## Bob randomly choose bases and measure the received qubits
bob_bases = generate_random_bits(n)

bob_bits = []

for i in range(n):

    measured_bit = measure_qubit(
        resent_qubits[i],
        bob_bases[i]
    )

    bob_bits.append(measured_bit)

In [114]:
# ======================================
# Public discussion and attack detection
# ======================================

matching_positions = []

for i in range(n):

    if alice_bases[i] == bob_bases[i]:
        matching_positions.append(i)

alice_sifted_key = []
bob_sifted_key = []

# Generate sifted keys using matching positions
for i in matching_positions:

    alice_sifted_key.append(alice_bits[i])
    bob_sifted_key.append(bob_bits[i])

# Alice and Bob publicly compare part of the sifted key
test_size = len(alice_sifted_key) // 2

alice_test_bits = alice_sifted_key[:test_size]
bob_test_bits = bob_sifted_key[:test_size]

errors = 0

# Count the number of mismatched bits
for i in range(len(alice_sifted_key)):

    if alice_sifted_key[i] != bob_sifted_key[i]:
        errors = errors + 1

# Calculate the error rate
error_rate = errors / len(alice_sifted_key)

# Threshold used to detect possible eavesdropping
threshold = 0.10

if error_rate > threshold:
    result = "Attack detected. Key rejected."
else:
    result = "No attack detected. Key accepted."

# Remaining bits form the final shared key
final_alice_key = alice_sifted_key[test_size:]
final_bob_key = bob_sifted_key[test_size:]

print("Alice bits: ", alice_bits)
print("Alice bases:", alice_bases)

print("Eve bases:  ", eve_bases)
print("Eve bits:   ", eve_bits)

print("Bob bases:  ", bob_bases)
print("Bob bits:   ", bob_bits)

print("Matching positions:", matching_positions)

print("Alice sifted key:", alice_sifted_key)
print("Bob sifted key:  ", bob_sifted_key)

print("Test size:", test_size)
print("Errors:", errors)
print("Error rate:", error_rate)
print(result)

print("Final Alice key:", final_alice_key)
print("Final Bob key:  ", final_bob_key)

Alice bits:  [0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1]
Alice bases: [1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0]
Eve bases:   [1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1]
Eve bits:    [0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1]
Bob bases:   [0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1]
Bob bits:    [0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1]
Matching positions: [2, 4, 8, 9, 11, 12, 13, 14, 16, 18]
Alice sifted key: [1, 0, 0, 1, 1, 0, 0, 0, 0, 0]
Bob sifted key:   [1, 0, 0, 1, 1, 0, 0, 1, 0, 0]
Test size: 5
Errors: 1
Error rate: 0.1
No attack detected. Key accepted.
Final Alice key: [0, 0, 0, 0, 0]
Final Bob key:   [0, 0, 1, 0, 0]


In [115]:
# The results show that Eve intercepts and measures the qubits before resending them to Bob.
# Eve's measurements may disturb the qubit states if she chooses the wrong basis, which can introduce errors into the sifted key.
# Because the basis choices are random, some runs may detect an attack while others may not, especially when the number of transmitted qubits is small.
# This follows the BB84 protocol from the lecture slides, where Alice and Bob compare matching-basis bits and use the error rate to detect possible eavesdropping.